# Exercise 03: Classification — Solution

Complete reference solution. Your exact numbers may vary slightly across library versions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

sns.set_style("whitegrid")
DATA_DIR = "../../../../resources/datasets"

## Part 1: Prepare the Data

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/heart-disease.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

print("Class balance:")
print(y.value_counts())
print(f"\nDisease rate: {y.mean():.1%}")

# ~54% positive, ~46% negative — reasonably balanced.
# Accuracy is a fair metric here, though we'll still look at
# precision/recall since this is a medical application.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## Part 2: Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

lr_preds = lr_model.predict(X_test)
print("Logistic Regression:")
print(classification_report(y_test, lr_preds))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, lr_preds)
plt.title("Logistic Regression — Confusion Matrix")
plt.show()

# Reading the matrix:
# - Top-left: correctly identified as no disease
# - Bottom-right: correctly identified as having disease
# - Top-right: false alarms (said disease, was healthy)
# - Bottom-left: MISSED CASES (said healthy, had disease) — the dangerous ones

## Part 3: Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
print("Random Forest:")
print(classification_report(y_test, rf_preds))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, rf_preds)
plt.title("Random Forest — Confusion Matrix")
plt.show()

In [ ]:
# Feature importances
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feature_importance.plot(kind="barh")
plt.title("Feature Importances — Heart Disease Prediction")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

# Top features typically include:
# - cp (chest pain type)
# - thalach (max heart rate)
# - ca (number of vessels colored by fluoroscopy)
# - oldpeak (ST depression)
# These align with medical knowledge about heart disease indicators.

## Part 4: Support Vector Machine

In [ ]:
# WITHOUT scaling
svm_unscaled = SVC(random_state=42)
svm_unscaled.fit(X_train, y_train)
svm_unscaled_preds = svm_unscaled.predict(X_test)
print("SVM WITHOUT scaling:")
print(f"Accuracy: {accuracy_score(y_test, svm_unscaled_preds):.3f}")

# This will likely show poor performance — possibly around 50-70%.
# Features like 'chol' (100-500) dominate over 'fbs' (0-1).

In [ ]:
# WITH scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # transform only — don't fit on test data!

svm_scaled = SVC(random_state=42)
svm_scaled.fit(X_train_scaled, y_train)
svm_preds = svm_scaled.predict(X_test_scaled)

print("SVM WITH scaling:")
print(classification_report(y_test, svm_preds))

# Notice the dramatic improvement. This is why scaling matters for SVMs.
# Common mistake: using fit_transform on test data. That would "cheat" by
# using test data statistics during scaling. Always fit on train, transform on test.

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, svm_preds)
plt.title("SVM (Scaled) — Confusion Matrix")
plt.show()

## Part 5: The Metrics That Matter

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "SVM (scaled)"],
    "Accuracy": [
        accuracy_score(y_test, lr_preds),
        accuracy_score(y_test, rf_preds),
        accuracy_score(y_test, svm_preds)
    ],
    "Precision": [
        precision_score(y_test, lr_preds),
        precision_score(y_test, rf_preds),
        precision_score(y_test, svm_preds)
    ],
    "Recall": [
        recall_score(y_test, lr_preds),
        recall_score(y_test, rf_preds),
        recall_score(y_test, svm_preds)
    ],
    "F1": [
        f1_score(y_test, lr_preds),
        f1_score(y_test, rf_preds),
        f1_score(y_test, svm_preds)
    ]
})

# Format as percentages for readability
for col in ["Accuracy", "Precision", "Recall", "F1"]:
    comparison[col] = comparison[col].map("{:.1%}".format)

print(comparison.to_string(index=False))

**Which metric matters most here?**

For a medical screening tool, **recall** is the priority. A false negative — telling a sick patient they're healthy — is far more dangerous than a false positive, which just means additional testing.

The model with the highest recall is the one that catches the most actual disease cases, even if it sometimes flags healthy patients for follow-up. In medicine, it's better to over-screen than to miss a diagnosis.

That said, if precision drops too low (too many false alarms), the tool becomes impractical — doctors stop trusting it and ignore the alerts. This is the precision/recall tradeoff in action.

## Part 6: Cross-Validation

In [ ]:
# For SVM, we need a Pipeline to handle scaling within cross-validation
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(random_state=42))
])

lr_cv = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X, y, cv=5, scoring="accuracy"
)
rf_cv = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X, y, cv=5, scoring="accuracy"
)
svm_cv = cross_val_score(svm_pipe, X, y, cv=5, scoring="accuracy")

print("5-Fold Cross-Validation (Accuracy):")
print(f"  Logistic Regression: {lr_cv.mean():.3f} (±{lr_cv.std():.3f})")
print(f"  Random Forest:       {rf_cv.mean():.3f} (±{rf_cv.std():.3f})")
print(f"  SVM (scaled):        {svm_cv.mean():.3f} (±{svm_cv.std():.3f})")

# The cross-validation winner may differ from the single-split winner.
# This is exactly why cross-validation matters — it gives a more reliable
# estimate of how the model will perform on truly unseen data.

## Reflection: The Consulting Scenario

**Recommendation for the healthcare client:**

Based on our analysis of 303 patient records with 13 clinical indicators, we built a heart disease prediction model that identifies at-risk patients with approximately 80-85% accuracy. The model is particularly good at catching actual cases — it misses very few patients who have heart disease, though it occasionally flags healthy patients for additional review.

The most important predictors are chest pain type, maximum heart rate achieved during exercise, and the number of major vessels visible in fluoroscopy imaging. These align well with established medical understanding, which gives us confidence the model is learning real patterns rather than noise.

**Our recommendation: use this as a screening assist tool, not a diagnostic tool.** It should flag patients for closer evaluation by a physician, not replace clinical judgment. A doctor who sees the model's flag can then apply their expertise, patient history, and additional tests to make the final call. This keeps the human in the loop for the high-stakes decision while letting the model surface cases that might otherwise be missed.

**Key limitations to be transparent about:**
- The training data is only 303 patients — a larger dataset would improve reliability
- The model doesn't account for patient history, family history, or lifestyle factors
- Performance may vary across demographic groups (age, sex, ethnicity) — we'd need to test for this before deployment
- This model should never be the sole basis for a diagnosis or treatment decision